In [36]:
import pypylon.pylon as pylon
import numpy as np
import cv2
import os

In [37]:
NUM_CAMERAS = 10

In [38]:
# setup demo environment with 10 cameras
os.environ["PYLON_CAMEMU"] = f"{NUM_CAMERAS}"

In [39]:
tlfactory = pylon.TlFactory.GetInstance()

In [40]:
# create a device filter for Pylon CamEmu devices
deviceinfo = pylon.DeviceInfo()
deviceinfo.SetDeviceClass("BaslerCamEmu")

# you could also set more device filters like e.g.:
# these are combined as AND 
# di.SetSerialNumber("2134234")

<pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d7201de3a0> >

In [41]:
devices = tlfactory.EnumerateDevices([deviceinfo,])

In [42]:
devices

(<pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d7201c7f30> >,
 <pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d738e46b80> >,
 <pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d7201de160> >,
 <pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d7201dca20> >,
 <pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d7201ddb00> >,
 <pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d7201de130> >,
 <pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d7201ddda0> >,
 <pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d7201de0d0> >,
 <pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDeviceInfo *' at 0x75d7201ddef0> >,
 <pypylon.pylon.DeviceInfo; proxy of <Swig Object of type 'Pylon::CDevice

In [43]:
camera_array = pylon.InstantCameraArray(NUM_CAMERAS)

In [44]:
for index, camera in enumerate(camera_array):
    camera.Attach(tlf.CreateDevice(devices[index]))

In [45]:
camera_array.Open()

In [46]:
# store a unique number for each camera to identify the incoming images
for index, camera in enumerate(camera_array):
    camera_serial = camera.DeviceInfo.GetSerialNumber()
    print(f"set context {index} for camera {camera_serial}")
    cam.SetCameraContext(index)

set context 0 for camera 0815-0000
set context 1 for camera 0815-0001
set context 2 for camera 0815-0002
set context 3 for camera 0815-0003
set context 4 for camera 0815-0004
set context 5 for camera 0815-0005
set context 6 for camera 0815-0006
set context 7 for camera 0815-0007
set context 8 for camera 0815-0008
set context 9 for camera 0815-0009


In [47]:
# set the exposure time for each camera
for index, camera in enumerate(camera_array):
    camera_serial = camera.DeviceInfo.GetSerialNumber()
    print(f"set Exposuretime {index} for camera {camera_serial}")
    cam.ExposureTimeRaw.Value = 10000

set Exposuretime 0 for camera 0815-0000
set Exposuretime 1 for camera 0815-0001
set Exposuretime 2 for camera 0815-0002
set Exposuretime 3 for camera 0815-0003
set Exposuretime 4 for camera 0815-0004
set Exposuretime 5 for camera 0815-0005
set Exposuretime 6 for camera 0815-0006
set Exposuretime 7 for camera 0815-0007
set Exposuretime 8 for camera 0815-0008
set Exposuretime 9 for camera 0815-0009


In [48]:
# wait for all cameras to grab 10 frames
frames_to_grab = 10
# store last framecount in array
frame_counts = [0]*NUM_CAMERAS

In [49]:
frame_counts

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [50]:
camera_array.StartGrabbing()
while True:
    with camera_array.RetrieveResult(1000) as result:
        if result.GrabSucceeded():
            image_number = result.ImageNumber
            camera_index = result.GetCameraContext()
            frame_counts[camera_index] = image_number
            print(f"cam #{camera_index}  image #{image_number}")
            
            # do something with the image ....
            
            # check if all cameras have reached 100 images
            if min(frame_counts) >= frames_to_grab:
                print( f"all cameras have acquired {frames_to_grab} frames")
                break
                
                
camera_array.StopGrabbing()

cam #0  image #1
cam #1  image #1
cam #2  image #1
cam #3  image #1
cam #0  image #2
cam #1  image #2
cam #2  image #2
cam #3  image #2
cam #4  image #1
cam #5  image #1
cam #6  image #1
cam #7  image #1
cam #8  image #1
cam #9  image #1
cam #0  image #3
cam #1  image #3
cam #2  image #3
cam #4  image #2
cam #3  image #3
cam #5  image #2
cam #6  image #2
cam #7  image #2
cam #8  image #2
cam #9  image #2
cam #0  image #4
cam #1  image #4
cam #2  image #4
cam #4  image #3
cam #3  image #4
cam #5  image #3
cam #6  image #3
cam #7  image #3
cam #8  image #3
cam #9  image #3
cam #0  image #5
cam #1  image #5
cam #4  image #4
cam #2  image #5
cam #5  image #4
cam #3  image #5
cam #6  image #4
cam #7  image #4
cam #8  image #4
cam #9  image #4
cam #0  image #6
cam #1  image #6
cam #4  image #5
cam #5  image #5
cam #2  image #6
cam #3  image #6
cam #6  image #5
cam #7  image #5
cam #8  image #5
cam #9  image #5
cam #0  image #7
cam #4  image #6
cam #5  image #6
cam #1  image #7
cam #6  image 

In [51]:
camera_array.Close()

In [52]:
frame_counts

[11, 11, 11, 11, 10, 10, 10, 10, 10, 10]